In [1]:
import sys
import json
import hashlib
from pathlib import Path
from datetime import datetime

import fitz
from IPython.display import display, Markdown

# Make config importable from notebooks/
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from config.settings import (
    PDF_PATH,
    RAW_IMAGES_DIR,
    MANIFESTS_DIR,
    OUTPUT_IMAGE_FORMAT,
    IMAGE_FILENAME_TEMPLATE,
    IMAGE_MANIFEST_PATH,
    OUTPUT_DIRS,
)

In [2]:
assert PDF_PATH.exists(), f"PDF not found: {PDF_PATH}"
for d in OUTPUT_DIRS:
    d.mkdir(parents=True, exist_ok=True)

doc = fitz.open(str(PDF_PATH))
manifest = []
skipped = 0

for page_num in range(len(doc)):
    page = doc[page_num]
    images = page.get_images(full=True)

    for img_idx, img_info in enumerate(images):
        xref = img_info[0]
        try:
            base = doc.extract_image(xref)
            image_bytes = base["image"]
            width = base["width"]
            height = base["height"]
            src_ext = base.get("ext", OUTPUT_IMAGE_FORMAT)
        except Exception:
            skipped += 1
            continue

        filename = IMAGE_FILENAME_TEMPLATE.format(
            page=page_num + 1,
            index=img_idx + 1,
            width=width,
            height=height,
            ext=OUTPUT_IMAGE_FORMAT,
        )
        out_path = RAW_IMAGES_DIR / filename

        # Always go through Pixmap to ensure RGB PNG output
        try:
            pix = fitz.Pixmap(doc, xref)
            # Convert CMYK or other colorspaces to RGB
            if pix.colorspace and pix.colorspace.n > 3:
                pix = fitz.Pixmap(fitz.csRGB, pix)
            # Drop alpha channel if present
            if pix.alpha:
                pix = fitz.Pixmap(pix, 0)  # 0 = drop alpha
            pix.save(str(out_path))
        except Exception:
            skipped += 1
            continue

        manifest.append({
            "filename": filename,
            "page": page_num + 1,
            "img_index": img_idx + 1,
            "xref": xref,
            "width": width,
            "height": height,
            "source_ext": src_ext,
            "file_size_bytes": out_path.stat().st_size,
            "md5": hashlib.md5(image_bytes).hexdigest(),
        })

doc.close()

# Save manifest
IMAGE_MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

# Summary
total_pages = fitz.open(str(PDF_PATH)).page_count
total_images = len(manifest)
total_mb = sum(m["file_size_bytes"] for m in manifest) / (1024 * 1024)

display(Markdown(f"""
### Image Extraction Summary

| Item | Value |
|------|-------|
| **PDF** | `{PDF_PATH.name}` |
| **Total pages** | {total_pages} |
| **Images extracted** | {total_images} |
| **Skipped** | {skipped} |
| **Total image size** | {total_mb:.2f} MB |
| **Output folder** | `{RAW_IMAGES_DIR.relative_to(project_root)}` |
| **Manifest** | `{IMAGE_MANIFEST_PATH.relative_to(project_root)}` |
| **Timestamp** | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} |
""".strip()))

### Image Extraction Summary

| Item | Value |
|------|-------|
| **PDF** | `Power SCADA Operation 2020 System Guide.pdf` |
| **Total pages** | 1314 |
| **Images extracted** | 1416 |
| **Skipped** | 0 |
| **Total image size** | 97.40 MB |
| **Output folder** | `SCADA-DIP\data\images\raw` |
| **Manifest** | `SCADA-DIP\data\manifests\image_manifest.json` |
| **Timestamp** | 2026-04-24 05:32:37 |

In [3]:
import shutil
from collections import Counter

import numpy as np
from PIL import Image

from config.settings import (
    RAW_IMAGES_DIR,
    FILTERED_KEEP_DIR,
    FILTERED_DISCARD_DIR,
    IMAGE_MANIFEST_PATH,
    FILTER_MANIFEST_PATH,
    MIN_WIDTH,
    MIN_HEIGHT,
    MIN_COMBINED_WIDTH,
    MIN_COMBINED_HEIGHT,
    BLANK_STD_THRESHOLD,
    BLANK_BRIGHTNESS_LOW,
    BLANK_BRIGHTNESS_HIGH,
)

# --- Helper functions ---

def is_too_small(w, h):
    if w < MIN_WIDTH or h < MIN_HEIGHT:
        return True
    if w < MIN_COMBINED_WIDTH and h < MIN_COMBINED_HEIGHT:
        return True
    return False


def is_blank_or_solid(img_path):
    img = Image.open(img_path).convert("L")
    pixels = np.array(img, dtype=np.float32)
    mean_val = pixels.mean()
    std_val = pixels.std()

    if std_val < BLANK_STD_THRESHOLD:
        if mean_val > BLANK_BRIGHTNESS_HIGH:
            return True, "blank_white"
        if mean_val < BLANK_BRIGHTNESS_LOW:
            return True, "blank_black"
        return True, "solid_color"
    return False, None


# --- Load manifest ---
assert IMAGE_MANIFEST_PATH.exists(), f"Manifest not found: {IMAGE_MANIFEST_PATH}"
raw_manifest = json.loads(IMAGE_MANIFEST_PATH.read_text(encoding="utf-8"))

FILTERED_KEEP_DIR.mkdir(parents=True, exist_ok=True)
FILTERED_DISCARD_DIR.mkdir(parents=True, exist_ok=True)

# --- Classify ---
seen_md5 = {}
filter_manifest = []
reason_counter = Counter()

for entry in raw_manifest:
    img_path = RAW_IMAGES_DIR / entry["filename"]
    if not img_path.exists():
        continue

    reasons = []
    w, h = entry["width"], entry["height"]
    md5 = entry["md5"]

    # Size check
    if is_too_small(w, h):
        reasons.append("too_small")

    # Duplicate check
    if md5 in seen_md5:
        reasons.append("duplicate")
    else:
        seen_md5[md5] = entry["filename"]

    # Blank/solid check
    if not reasons:
        is_blank, blank_reason = is_blank_or_solid(img_path)
        if is_blank:
            reasons.append(blank_reason)

    # Decision
    decision = "discard" if reasons else "keep"
    dest_dir = FILTERED_DISCARD_DIR if decision == "discard" else FILTERED_KEEP_DIR
    shutil.copy2(img_path, dest_dir / entry["filename"])

    for r in reasons:
        reason_counter[r] += 1

    filter_manifest.append({
        **entry,
        "decision": decision,
        "reasons": reasons,
    })

# Save filter manifest
FILTER_MANIFEST_PATH.write_text(json.dumps(filter_manifest, indent=2), encoding="utf-8")

# --- Summary ---
total = len(filter_manifest)
keep_count = sum(1 for m in filter_manifest if m["decision"] == "keep")
discard_count = total - keep_count
reduction_pct = (discard_count / total * 100) if total else 0

reason_lines = "\n".join(
    f"| {reason} | {count} |" for reason, count in reason_counter.most_common()
)

# Examples per discard reason (max 3 each)
example_sections = ""
for reason in reason_counter:
    examples = [
        m for m in filter_manifest
        if m["decision"] == "discard" and reason in m["reasons"]
    ][:3]
    rows = "\n".join(
        f"| `{e['filename']}` | {e['page']} | {e['width']} | {e['height']} | {e['file_size_bytes'] / 1024:.1f} KB |"
        for e in examples
    )
    example_sections += f"""
**{reason}** (showing up to 3)

| Filename | Page | W | H | Size |
|----------|------|---|---|------|
{rows}

"""

display(Markdown(f"""
### Image Filtering Summary

| Item | Value |
|------|-------|
| **Total raw images** | {total} |
| **Keep** | {keep_count} |
| **Discard** | {discard_count} |
| **Reduction** | {reduction_pct:.1f}% |
| **Keep folder** | `{FILTERED_KEEP_DIR.relative_to(project_root)}` |
| **Discard folder** | `{FILTERED_DISCARD_DIR.relative_to(project_root)}` |
| **Filter manifest** | `{FILTER_MANIFEST_PATH.relative_to(project_root)}` |

#### Discard Reasons

| Reason | Count |
|--------|-------|
{reason_lines}

#### Discard Examples
{example_sections}
""".strip()))

### Image Filtering Summary

| Item | Value |
|------|-------|
| **Total raw images** | 1416 |
| **Keep** | 738 |
| **Discard** | 678 |
| **Reduction** | 47.9% |
| **Keep folder** | `SCADA-DIP\data\filtered\keep` |
| **Discard folder** | `SCADA-DIP\data\filtered\discard` |
| **Filter manifest** | `SCADA-DIP\data\manifests\filter_manifest.json` |

#### Discard Reasons

| Reason | Count |
|--------|-------|
| too_small | 536 |
| duplicate | 458 |
| blank_white | 1 |
| blank_black | 1 |

#### Discard Examples

**too_small** (showing up to 3)

| Filename | Page | W | H | Size |
|----------|------|---|---|------|
| `page0001_img001_w0200_h0059.png` | 1 | 200 | 59 | 3.4 KB |
| `page0001_img002_w0200_h0059.png` | 1 | 200 | 59 | 1.1 KB |
| `page0003_img002_w0100_h0105.png` | 3 | 100 | 105 | 3.3 KB |


**duplicate** (showing up to 3)

| Filename | Page | W | H | Size |
|----------|------|---|---|------|
| `page0003_img005_w0125_h0108.png` | 3 | 125 | 108 | 1.0 KB |
| `page0033_img001_w0125_h0108.png` | 33 | 125 | 108 | 1.0 KB |
| `page0033_img002_w0125_h0108.png` | 33 | 125 | 108 | 0.8 KB |


**blank_white** (showing up to 3)

| Filename | Page | W | H | Size |
|----------|------|---|---|------|
| `page0049_img001_w1270_h0572.png` | 49 | 1270 | 572 | 1.9 KB |


**blank_black** (showing up to 3)

| Filename | Page | W | H | Size |
|----------|------|---|---|------|
| `page0075_img002_w0200_h0200.png` | 75 | 200 | 200 | 0.2 KB |

In [4]:
from config.settings import (
    FILTERED_KEEP_DIR,
    FILTERED_DISCARD_DIR,
    IMAGE_MANIFEST_PATH,
    FILTER_MANIFEST_PATH,
)

# Load original manifest
assert IMAGE_MANIFEST_PATH.exists(), f"Not found: {IMAGE_MANIFEST_PATH}"
raw_manifest = json.loads(IMAGE_MANIFEST_PATH.read_text(encoding="utf-8"))

# Build lookup sets from actual folder contents
keep_files = {f.name for f in FILTERED_KEEP_DIR.iterdir() if f.is_file()}
discard_files = {f.name for f in FILTERED_DISCARD_DIR.iterdir() if f.is_file()}

# Rebuild filter manifest
filter_manifest = []
unmatched = []

for entry in raw_manifest:
    fname = entry["filename"]
    if fname in keep_files:
        decision = "keep"
    elif fname in discard_files:
        decision = "discard"
    else:
        decision = "missing"
        unmatched.append(fname)

    filter_manifest.append({**entry, "decision": decision, "source": "manual_review"})

# Save
FILTER_MANIFEST_PATH.write_text(json.dumps(filter_manifest, indent=2), encoding="utf-8")

# Summary
keep_count = sum(1 for m in filter_manifest if m["decision"] == "keep")
discard_count = sum(1 for m in filter_manifest if m["decision"] == "discard")
missing_count = len(unmatched)

summary = f"""
### Filter Manifest Rebuilt (Manual Review)

| Item | Value |
|------|-------|
| **Total in manifest** | {len(filter_manifest)} |
| **Keep** | {keep_count} |
| **Discard** | {discard_count} |
| **Missing from both folders** | {missing_count} |
| **Manifest saved** | `{FILTER_MANIFEST_PATH.relative_to(project_root)}` |
"""

if unmatched:
    examples = "\n".join(f"| `{f}` |" for f in unmatched[:10])
    summary += f"""
#### Missing Files (up to 10)

| Filename |
|----------|
{examples}
"""

display(Markdown(summary.strip()))

### Filter Manifest Rebuilt (Manual Review)

| Item | Value |
|------|-------|
| **Total in manifest** | 1416 |
| **Keep** | 866 |
| **Discard** | 550 |
| **Missing from both folders** | 0 |
| **Manifest saved** | `SCADA-DIP\data\manifests\filter_manifest.json` |